# 03 Baseline Models

Trains classical baselines for comparison against the proposed embedding and FL model.

In [1]:
from pathlib import Path
import os
import sys

ABS_PROJECT_ROOT = Path('/Volumes/LocalDrive1/onlyscholar-projects/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION')
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/projects/implementation/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION')


def find_project_root():
    candidates = []
    env_root = os.environ.get('ECOTRUST_PROJECT_ROOT')
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend([Path.cwd(), Path.cwd().parent, DRIVE_PROJECT_ROOT, ABS_PROJECT_ROOT])
    for base in list(candidates):
        candidates.extend(base.parents)
    seen = set()
    for candidate in candidates:
        expanded = candidate.expanduser()
        resolved = expanded.resolve() if expanded.exists() else expanded
        if str(resolved) in seen:
            continue
        seen.add(str(resolved))
        if (resolved / 'configs' / 'project.json').exists() and (resolved / 'src' / 'ecotrust').exists():
            return resolved
    raise RuntimeError(
        'Cannot find EcoTrust-FLIDS IMPLEMENTATION. In Colab, mount Drive and set ECOTRUST_PROJECT_ROOT to '
        '/content/drive/MyDrive/projects/implementation/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION.'
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
if str(PROJECT_ROOT).startswith('/content/drive') and (PROJECT_ROOT / 'configs' / 'project_colab.json').exists():
    CONFIG_PATH = PROJECT_ROOT / 'configs' / 'project_colab.json'
else:
    CONFIG_PATH = PROJECT_ROOT / 'configs' / 'project.json'
RUN_MODE = 'full'  # change to 'full' after smoke run passes
DATASETS = ['Edge-IIoTset', 'TON_IoT', 'CICIoT2023']
print('PROJECT_ROOT:', PROJECT_ROOT)
print('CONFIG_PATH:', CONFIG_PATH)

from ecotrust.config import load_config
from ecotrust.data.preprocess import load_preprocessed
from ecotrust.models.baselines import train_baselines
from ecotrust.evaluation.reports import write_metrics_report
cfg = load_config(CONFIG_PATH)
for dataset in DATASETS:
    print('Baselines', dataset)
    prep = cfg.results_root / dataset / 'preprocessed' / RUN_MODE
    X_train, X_test, y_train, y_test = load_preprocessed(prep)
    out = cfg.results_root / dataset / 'baselines' / RUN_MODE
    metrics = train_baselines(X_train, y_train, X_test, y_test, out, seed=cfg.seed)
    write_metrics_report(out, f'{dataset} baseline models', metrics)
    print(metrics)

PROJECT_ROOT: /content/drive/MyDrive/projects/implementation/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION
CONFIG_PATH: /content/drive/MyDrive/projects/implementation/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION/configs/project_colab.json
Baselines Edge-IIoTset
{'logistic_regression': {'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'roc_auc': 1.0}, 'random_forest': {'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'roc_auc': 1.0}}
Baselines TON_IoT
{'logistic_regression': {'accuracy': 0.8937197019813684, 'precision': 0.7757894995938661, 'recall': 0.9785463690272905, 'f1': 0.8654511505299577, 'roc_auc': 0.9684718613844993}, 'random_forest': {'accuracy': 0.9993709941545836, 'precision': 0.9984805730410246, 'recall': 0.9997205749945668, 'f1': 0.9991001892705328, 'roc_auc': 0.9999976352365281}}
Baselines CICIoT2023
{'logistic_regression': {'accuracy': 0.97724, 'precision': 0.9998533263488738, 'recall': 0.9768470183627096, 'f1': 0.9882162900987843, 'roc_auc': 0.995194752445